# Practical 3 — Superstore discounts and profit

R Jupyter version of the course material. Select an **R** kernel in Positron and run cells from top to bottom, starting with the setup cell. Keep the complete course folder, data and images together.

Generated from the matching QMD; make teaching edits in the QMD and regenerate. The published HTML/PDF retains the full course styling.


In [ ]:
# Notebook sessions may start in the notebook folder.
# Find the opened course folder so the original data/... paths work unchanged.
.course_root <- normalizePath(getwd(), winslash = "/")
while (!file.exists(file.path(.course_root, "_quarto.yml")) ||
       !dir.exists(file.path(.course_root, "data"))) {
  .course_parent <- dirname(.course_root)
  if (identical(.course_parent, .course_root)) {
    stop("Open the complete course folder containing _quarto.yml and data/.")
  }
  .course_root <- .course_parent
}
setwd(.course_root)
rm(.course_root)
if (exists(".course_parent", inherits = FALSE)) rm(.course_parent)


In [ ]:
#| label: association-import
source("lessons/_includes/course-tools.R")
lines <- read.csv("data/business/superstore-lines.csv",
                  stringsAsFactors = FALSE, colClasses = c(postal_code = "character"))
orders <- make_orders(lines)
lines <- lines %>% mutate(
  line_margin_pct = 100 * profit / sales,
  loss_line = profit < 0,
  discount_band = cut(discount, breaks = c(-Inf, 0, .2, Inf),
    labels = c("None", "Above 0 to 20%", "Above 20%"), ordered_result = TRUE))
stopifnot(!anyNA(lines$discount_band), all(lines$sales > 0))


- `data/business/superstore-lines.csv`; guide: `data/business/README.md`.
- A 20% discount belongs to the middle band; zero belongs to **None**.
- These bands are analyst-defined categories, not source classifications.
- A line’s margin is profit divided by its recorded sales.
- We do not construct a discount-adjusted revenue or refund measure from undocumented accounting assumptions.

## 3. Compare categories fairly


In [ ]:
#| fig-alt: "Overall category profit margins; Furniture has a much thinner margin than the other categories."
category_results <- lines %>% group_by(category) %>%
  summarise(line_count = n(), total_sales = sum(sales), total_profit = sum(profit),
            median_line_profit = median(profit), loss_line_pct = 100 * mean(loss_line),
            .groups = "drop") %>%
  mutate(overall_margin_pct = 100 * total_profit / total_sales)
knitr::kable(category_results, digits = 2)
ggplot(category_results, aes(category, overall_margin_pct)) +
  geom_col(fill = petra_ink) +
  labs(x = NULL, y = "Total profit / total sales (%)")


- Furniture has substantial sales but a much thinner overall margin in this extract.
- Category totals describe business contribution; median line profit describes a typical line.
- The loss-line rate uses lines as its denominator.
- A category can contain both profitable and loss-making subcategories.

### Why an average of percentages can mislead


In [ ]:
margin_comparison <- lines %>% group_by(category) %>%
  summarise(equal_line_mean_margin_pct = mean(line_margin_pct),
            sales_weighted_margin_pct = 100 * sum(profit) / sum(sales), .groups = "drop")
knitr::kable(margin_comparison, digits = 2)


- The first gives every line equal weight.
- The second gives each dollar of sales equal weight.
- Use the second to report the overall category margin.

## 4. Examine discount groups


In [ ]:
#| fig-alt: "Boxplots of line margins across three discount bands, with negative margins retained."
discount_results <- lines %>% group_by(discount_band) %>%
  summarise(line_count = n(), total_sales = sum(sales), total_profit = sum(profit),
            median_line_profit = median(profit), loss_line_pct = 100 * mean(loss_line),
            .groups = "drop") %>%
  mutate(overall_margin_pct = 100 * total_profit / total_sales)
knitr::kable(discount_results, digits = 2)
ggplot(lines, aes(discount_band, line_margin_pct)) +
  geom_hline(yintercept = 0, colour = "#A3A3A3") +
  geom_boxplot(fill = petra_accent, outlier.alpha = .25) +
  labs(x = "Recorded line discount", y = "Line profit / line sales (%)")


- Compare medians, spread, group sizes and weighted margins.
- The chart gives each line equal weight; the overall margin table does not.
- A negative line margin is possible and is not automatically a bad import.
- Discounts, prices and profit are connected accounting quantities; an association is expected to reflect more than customer response alone.

### Keep product mix visible


In [ ]:
#| fig-alt: "Overall profit margins by discount band within each category; zero discount is distinct from positive discounts up to twenty percent."
within_category <- lines %>% group_by(category, discount_band) %>%
  summarise(line_count = n(), margin_pct = 100 * sum(profit) / sum(sales),
            .groups = "drop")
knitr::kable(within_category, digits = 2)
ggplot(within_category, aes(discount_band, margin_pct)) +
  geom_col(fill = petra_ink) + facet_wrap(~ category) +
  scale_x_discrete(labels = c("None" = "0%", "Above 0 to 20%" = "(0,20%]",
                               "Above 20%" = ">20%")) +
  labs(x = "Discount band", y = "Group overall margin (%)")


- Stratification compares discount bands within each category.
- It reduces one source of mixing, but does not remove all confounding.
- It does not establish what would happen if the manager changed discounts.

## 5. Association at order level

> Are higher-sales orders also higher-profit orders?


In [ ]:
#| fig-alt: "Order sales versus order profit, including large positive and negative profit values."
ggplot(orders, aes(sales, profit)) +
  geom_hline(yintercept = 0, colour = "#A3A3A3") +
  geom_point(alpha = .35, colour = petra_ink) +
  labs(x = "Order sales ($)", y = "Order profit ($)")

correlations <- orders %>% group_by(segment) %>%
  summarise(orders = n(), pearson_r = cor(sales, profit),
            spearman_r = cor(sales, profit, method = "spearman"), .groups = "drop")
knitr::kable(correlations, digits = 3)


- Pearson measures linear association; Spearman uses ranks.
- Large observations can strongly influence Pearson correlation.
- Segment correlations can differ from the overall correlation.
- Correlation does not show the distribution of losses or establish a causal effect.
- Repeated customers matter if we later try to model orders as independent future observations.

## 6. Turn a pattern into a decision question

**Supported wording:**

- “In this extract, the highest discount band has weaker recorded profitability.”
- “Furniture’s overall margin warrants a closer subcategory review.”

**Before proposing a discount change, obtain:**

- Product cost and accounting definitions.

- Promotion timing and reasons for discount assignment.

- Comparable product groups and customer demand evidence.

- A planned experiment or another defensible causal design.

- Do not claim that removing discounts would reproduce the undiscounted group’s margin.

- Do not describe a line-count loss rate as the percentage of customers losing money.

## 7. Independent commercial briefing

1.  Rank subcategories by total profit and by overall margin. Include sales and line counts.
2.  Select one weak-performing subcategory and compare its discount bands.
3.  Find its three largest loss-making lines; retain them in the main analysis.
4.  Compare order-level loss percentages across customer segments.
5.  Produce two charts answering different questions.
6.  Write a recommendation separating an observed finding, a plausible explanation and evidence still needed.


**Manual setup — run once in the R Console if needed.** This cell is excluded from Run All.

```r
#| eval: false
# Use lines for product/discount questions and orders for basket questions.
# A group loss percentage is 100 * mean(profit < 0).
# Calculate rates before overwriting any column with a group total.
```


**Submission:**

- One compact results table.
- Two labelled charts with units and denominators.
- Five short briefing bullets, including one limitation and one proposed follow-up.

## 8. Move from description to inference

- [Part 4](https://msfasha.github.io/307312-Applied-Statistics/lessons/04-statistical-inference-and-estimation.html) covers estimation, hypothesis testing and regression; Part 3 provides the probability foundations.
- The descriptive evidence here concerns the complete fixed extract; p-values are not required to describe its totals.
- `data/business/README.md`.
